# Agun Panjika: validation notebook

Reads the pipeline outputs (`python -m pipeline.run_all`) and reproduces every validation number and figure in the README and the app. The same figures are written to `docs/figures/` by `python notebooks/figures.py`.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
P = ROOT / 'data' / 'processed'
import figures as F

## 1. Counts match published totals

In [ ]:
det = pd.read_parquet(P / 'detections.parquet', columns=['sensor', 'year'])
print('MODIS 2003-2021:', int(((det.sensor == 'modis') & det.year.between(2003, 2021)).sum()), '(Farukh et al. 2023: 54,669)')
y = det.year.between(2019, 2024)
print('NOAA-20 2019-2024:', int(((det.sensor == 'noaa20') & y).sum()), ' S-NPP:', int(((det.sensor == 'snpp') & y).sum()))

## 2. The problem: the stitched record jumps in 2012

In [ ]:
F.fig_stitched_jump(); F.fig_monthly_ratio()

## 3. Leave-one-year-out validation (2012-2024)

In [ ]:
v = json.loads((P / 'validation.json').read_text())
pd.DataFrame(v['summary']).set_index('model').round(2)

In [ ]:
pd.DataFrame(v['per_year']).pivot(index='year', columns='model', values='mae_week').round(1)

In [ ]:
print('80% interval coverage (nested LOYO):', v['interval_coverage'])
F.fig_validation()

## 4. Independent check: ESA Fire_cci burned area

In [ ]:
p = json.loads((P / 'partner.json').read_text())
print(p['jump_2012'], p['corr_log_annual'], p['corr_log_annual_estimated_years'], p['corr_seasonal_cycle'])
F.fig_partner()

## 5. Handover bridge

In [ ]:
b = json.loads((P / 'bridge_factors.json').read_text())
{s: (b[s]['form'], b[s]['loyo_score']) for s in ('noaa20', 'noaa21')}

## 6. Early warning hindcast

In [ ]:
f = json.loads((P / 'forecast.json').read_text())
{k: {m: f['products'][k]['skill'][m] for m in ('bss_fire_only', 'bss_full', 'auc_fire_only', 'auc_full', 'mae_fire_only', 'mae_full', 'mae_clim')} for k in ('next2', 'rest')}

In [ ]:
F.fig_outlook_skill(); F.fig_harmonised(); F.fig_calendar()

## 7. Kiln filter spot checks

Cells judged on Esri World Imagery; Y = industry/kiln/power/gas/landfill, U = unclear, N = vegetation.

In [ ]:
for r in (1, 2, 3):
    d = pd.read_csv(ROOT / 'docs' / f'kiln_spotcheck_round{r}.csv')
    print(r, d.is_kiln_or_industry.value_counts().to_dict())